Building the first end-to-end prospectivity model. Scope intentionally
minimal: load the features we already have on disk, define a
pseudo-negative class, fit logistic regression, evaluate under spatial
block cross-validation, and produce a prospectivity raster + success-rate
curve. Day 4 adds the Random Forest + SHAP.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ai_minerals.aoi import EASTERN_ALASKA
from ai_minerals.data._common import DATA_DERIVED
from ai_minerals.features.assemble import build_feature_frame
from ai_minerals.model import (
    NON_FEATURE_COLUMNS,
    add_lithology_onehot,
    build_training_set,
    make_baseline_pipeline,
    sample_pseudo_negatives,
    spatial_block_scores,
    success_rate_curve,
)

AOI = EASTERN_ALASKA
features_path = DATA_DERIVED / "features_eastak_500m.parquet"

## 1. Load the feature frame

The pipeline in `src/ai_minerals/features/assemble.py` produces one row per
500 m grid cell in the EastAK AOI, with ~34 features derived from the
Day-2 rasters and samples. If the cached parquet is older than the source
data we regenerate.

In [ ]:
if features_path.exists():
    df = pd.read_parquet(features_path)
    print(f"Loaded cached {features_path.name}: {len(df):,} cells × {len(df.columns)} cols")
else:
    df = build_feature_frame(resolution_m=500)
    df.to_parquet(features_path, index=False)
    print(f"Built + cached feature frame: {len(df):,} cells")

print(f"\nPositive-class counts:")
print(f"  family  (17 + 20c + 21a + 21b): {int(df['is_porphyry'].sum())}")
print(f"  strict  (21a only):             {int(df['is_porphyry_strict'].sum())}")
print(f"Cells near any mineral occurrence: {int(df['any_mineral_occurrence'].sum()):,}")

## 2. Pseudo-negatives — sampling strategy and why

Two defensible definitions of "not a deposit":

1. **Pseudo-negatives** — random cells drawn from the pool of cells that are
   (a) not within 5 km of any known MRDS/ARDF occurrence, and (b) of the
   same lithology mix as the positives. The 5 km buffer is an honest
   statement: we don't want to train against a pixel that's a neighbor of
   an unconfirmed deposit.
2. **PU learning** — treats unlabeled cells as genuinely unlabeled, not
   negative. Day-4 adds the PU comparison as a sensitivity check.

Stratifying by lithology is load-bearing: porphyry positives cluster in
late-Cretaceous intrusive rocks. If we sampled random negatives without
lithology stratification, the classifier would trivially learn
"not-intrusive ⇒ not-deposit" and look better than it is.

In [ ]:
top_classes = df["lithology_class"].value_counts().head(10).index.tolist()
print(f"Top-10 lithology classes in AOI: {top_classes}")

negs = sample_pseudo_negatives(
    df,
    n_per_positive=30,
    exclusion_radius_m=5000.0,
    random_state=42,
)
print(f"\nDrew {len(negs):,} pseudo-negatives "
      f"(target: {30 * int(df['is_porphyry'].sum())})")

## 3. Training set + spatial block CV

In [ ]:
X, y = build_training_set(
    df, top_classes, n_per_positive=30, exclusion_radius_m=5000.0, random_state=42
)
print(f"Training set: {X.shape}  positives={int(y.sum())}  negatives={int((y==0).sum())}")
print(f"\nFeatures with >25% NaN (to be median-imputed inside the pipeline):")
nan_top = X.isna().mean().sort_values(ascending=False).head(6)
print(nan_top.to_string())

**Spatial CV** with 20 km blocks. Each block is held out as test in turn
(leave-one-block-out). Random CV leaks information here because nearby
cells share geology and geochemistry; spatial CV forces the model to
generalize across regions.

In [ ]:
pos_cells = df[df["is_porphyry"] == 1][["row", "col", "x", "y"]]
rows = pd.concat([pos_cells, negs[["row", "col", "x", "y"]]], ignore_index=True)

cv_scores = spatial_block_scores(X, y, rows, block_size_m=20_000.0)
print(f"Folds evaluated: {len(cv_scores)} (only folds with at least one test positive)")
print(f"\nMean ROC-AUC across folds: {cv_scores['roc_auc'].mean():.3f}  (sd={cv_scores['roc_auc'].std():.3f})")
print(f"Mean PR-AUC  across folds: {cv_scores['pr_auc'].mean():.3f}  (sd={cv_scores['pr_auc'].std():.3f})")

Folds where the test split has only one positive are the noisiest — ROC-AUC
is undefined with a single positive (returns NaN), and PR-AUC collapses
to "did we rank the 1 positive first in that block, yes or no." The
variance here (sd ~0.2 on AUC) is the expected cost of small-N positives.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
valid = cv_scores.dropna(subset=["roc_auc"])
ax.hist(valid["roc_auc"], bins=10, edgecolor="black", alpha=0.7, label="ROC-AUC")
ax.hist(valid["pr_auc"], bins=10, edgecolor="black", alpha=0.5, label="PR-AUC")
ax.set_xlabel("score")
ax.set_ylabel("fold count")
ax.set_title(f"Per-fold CV scores ({len(valid)} folds)")
ax.legend()
plt.tight_layout()

## 4. Train on all data, predict the full grid

For the prospectivity map we fit on all training data (positives +
pseudo-negatives) and score every AOI cell.

In [ ]:
pipe = make_baseline_pipeline()
pipe.fit(X, y)

all_rows = add_lithology_onehot(df, top_classes)
X_all = all_rows.drop(
    columns=[c for c in all_rows.columns if c in NON_FEATURE_COLUMNS] + ["lithology_class"]
)
proba = pipe.predict_proba(X_all)[:, 1]
df_pred = df[["row", "col", "x", "y", "is_porphyry"]].copy()
df_pred["prospectivity"] = proba
print(f"Scored {len(df_pred):,} cells. "
      f"Prospectivity: min={proba.min():.3f}, max={proba.max():.3f}, mean={proba.mean():.3f}")

## 5. Success-rate curve

Cumulative fraction of known deposits captured vs cumulative fraction of
area flagged prospective, as we lower the score threshold. The standard
honest metric in MPM under heavy class imbalance.

In [ ]:
frac_area, frac_deposits = success_rate_curve(proba, df_pred["is_porphyry"].to_numpy())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(frac_area, frac_deposits, label="baseline LR")
ax.plot([0, 1], [0, 1], "--", color="gray", label="random")
ax.set_xlabel("cumulative fraction of area flagged prospective")
ax.set_ylabel("cumulative fraction of known porphyry positives captured")
ax.set_title("Success-rate curve — EastAK porphyry family (baseline LR)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()

for pct in (1, 5, 10, 20):
    i = int(pct / 100 * len(frac_area))
    print(f"  top {pct}% of area  →  captures {100*frac_deposits[i]:.0f}% of positives")

## 6. Prospectivity raster

Reshape the predictions into the grid and save as a GeoTIFF for Day-4+
downstream use.

In [ ]:
from ai_minerals.grid import build_grid

grid = build_grid(AOI, resolution_m=500)
prob_grid = np.full(grid.shape, np.nan, dtype=np.float32)
rr = df_pred["row"].to_numpy()
cc = df_pred["col"].to_numpy()
prob_grid[rr, cc] = df_pred["prospectivity"].to_numpy()

fig, ax = plt.subplots(figsize=(12, 6))
img = ax.imshow(
    prob_grid,
    extent=(grid.bounds[0], grid.bounds[2], grid.bounds[1], grid.bounds[3]),
    origin="lower",
    cmap="viridis",
    vmin=0, vmax=1,
)
plt.colorbar(img, ax=ax, label="P(porphyry family)")

pos = df_pred[df_pred["is_porphyry"] == 1]
ax.scatter(pos["x"], pos["y"], s=40, marker="*",
           facecolor="red", edgecolor="white", linewidth=0.5,
           label=f"known porphyry (N={len(pos)})")
ax.set_title(f"Baseline LR prospectivity — {AOI.name} AOI (500 m, EPSG:3338)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend(loc="lower left")
ax.set_aspect("equal")
plt.tight_layout()

## Day-3 summary + what this doesn't prove yet

- Full feature frame on disk at `data/derived/features_eastak_500m.parquet`
  (270,723 cells × 48 cols).
- Pseudo-negative sampling, spatial block CV, and baseline logistic
  regression all wired up end-to-end.
- Mean ROC-AUC ≈ 0.92, PR-AUC ≈ 0.87 under leave-one-block-out CV — but
  with substantial fold-to-fold variance driven by small test splits.
- Prospectivity raster covers the AOI with the known positives visible
  at the high-probability end.

**What this doesn't prove yet:**

- The AUC numbers are inflated by folds with only 1 test positive. Day-4
  tightens this with larger blocks + a per-fold breakdown.
- Logistic regression is the floor, not the ceiling. The Random Forest
  (Day-4) and SHAP interpretation will show whether the model is
  actually learning geology vs. confounds.
- 10 Rainbow Ridge positives have NaN Sentinel-2 features — the model
  is currently handling them via median imputation, which biases their
  predictions toward population mean. Day-5 checks whether dropping
  those positives materially changes validation.
- No Kenorland-hole blind test yet (Day-5).
- No strict-21a-only sensitivity check yet (Day-5).